## Guardrails
Guardrails w LangChain to mechanizmy pilnujące, aby odpowiedzi modelu spełniały określone wymagania techniczne lub formatowe.
Na przykład JsonValidityEvaluator sprawdza, czy zwrócony tekst jest poprawnym JSON-em – jeśli struktura jest niepoprawna, evaluator zwróci błąd.
Dzięki temu można automatycznie wykrywać i odrzucać odpowiedzi, które nie nadają się do dalszego przetwarzania.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mzarnecki/course_llm_agent_apps_with_langchain_and_langgraph/blob/main/notebooks/pl/6_5_Guardrails.ipynb)

| | |
|---|---|
| 🎯 **Czego się nauczysz** | Guardrails: walidacja JSON i regex, modele zapasowe (fallback), przycinanie kontekstu i limit długości odpowiedzi. |
| 🔑 **Klucze API** | `OPENAI_API_KEY` |
| 💲 **Koszt API** | bardzo niski – kilka krótkich zapytań |
| ⏱️ **Czas wykonania** | < 30 s |

### JSON Format Validator
Sprawdza, czy wygenerowana odpowiedź jest poprawnym JSON-em

In [ ]:
# Zainstaluj bezpośrednie zależności używane w tym notebooku.
# Pomijane, gdy brak pip, np. w środowisku `uv sync`, które ma je już zainstalowane (zob. README).
import importlib.util
if importlib.util.find_spec("pip"):
    %pip install -qU langchain-classic langchain-openai python-dotenv
# Google Colab: dodaj OPENAI_API_KEY w panelu Secrets (🔑) i zezwól notatnikowi na dostęp.
# Odkomentuj te linie, aby wczytać klucz z Colab Secrets do bieżącej sesji:
# from google.colab import userdata
# import os
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# Lokalnie umieść OPENAI_API_KEY=... w pliku .env; nie commituj klucza.

In [2]:
from langchain_classic.evaluation import JsonValidityEvaluator

evaluator = JsonValidityEvaluator()
# print(evaluator.evaluate_strings(prediction='{"x": 1}'))      # poprawne
print(evaluator.evaluate_strings(prediction='{x: 1}'))        # błąd


{'score': 0, 'reasoning': 'Expecting property name enclosed in double quotes: line 1 column 2 (char 1)'}


### JsonEqualityEvaluator
Sprawdza równość JSON-ów po sparsowaniu (kolejność kluczy w JSON nie ma znaczenia)

In [3]:
from langchain_classic.evaluation import JsonEqualityEvaluator

evaluator = JsonEqualityEvaluator()
print(evaluator.evaluate_strings(
    prediction='{"a":1,"b":[2,3]}',
    reference='{"b":[2,3],"a":2}',
))


{'score': False}


### RegexMatchEvaluator
Sprawdza dopasowanie do wyrażenia regularnego

In [4]:
from langchain_classic.evaluation import RegexMatchStringEvaluator

evaluator = RegexMatchStringEvaluator()
result = evaluator.evaluate_strings(
    prediction="Order ID: ABC-1234",
    reference=r"^Order ID: [A-Z]{3}-\d{4}$",
)
print(result['score'])

iter = 3
while result['score'] < 1.0 and iter > 0:
    iter -= 1
    print('run model once more')

1


### Fallback Messages Validator
Kiedy główny model zawiedzie (rate limit / błąd) automatycznie użyj modelu zapasowego.

In [5]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()

primary = ChatOpenAI(model="gpt-6-lunaS", max_retries=0)  # S w gpt-6-lunaS jest celową literówką
backup  = ChatOpenAI(model="gpt-5-mini")

chain = primary.with_fallbacks([backup])

print(chain.invoke("Opisz w 1 zdaniu Python."))


content='Python to wysokopoziomowy, interpretowany i dynamicznie typowany język programowania o czytelnej składni, bogatej bibliotece standardowej i rozbudowanym ekosystemie pakietów, używany m.in. do tworzenia aplikacji webowych, analizy danych, automatyzacji i uczenia maszynowego.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 341, 'prompt_tokens': 15, 'total_tokens': 356, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EabPMeRwEeSl067gtaMICT1UmevFq', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a152e8-3511-7fd2-a3ed-5d2bb554477f-0' tool_calls=

### Token Limit
Śledzenie i przycinanie historii do limitu tokenów, żeby nie przekroczyć kontekstu modelu.

In [6]:
import json
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.messages.utils import trim_messages, count_tokens_approximately
from langchain_openai import ChatOpenAI

messages = [
    SystemMessage(content="Jesteś pomocnym asystentem."),
    HumanMessage(content="(tu długa historia rozmowy / wiele wiadomości...)"),
]

trimmed = trim_messages(
    messages,
    strategy="last",
    token_counter=count_tokens_approximately,
    max_tokens=256,
    start_on="human",
    include_system=True,
)

llm = ChatOpenAI(model="gpt-6-luna")
print(json.dumps(llm.invoke(trimmed).response_metadata, indent=4))


{
    "token_usage": {
        "completion_tokens": 139,
        "prompt_tokens": 32,
        "total_tokens": 171,
        "completion_tokens_details": {
            "accepted_prediction_tokens": 0,
            "audio_tokens": 0,
            "reasoning_tokens": 63,
            "rejected_prediction_tokens": 0,
            "text_tokens": null
        },
        "prompt_tokens_details": {
            "audio_tokens": 0,
            "cache_write_tokens": 0,
            "cached_tokens": 0,
            "image_tokens": null,
            "text_tokens": null
        }
    },
    "model_provider": "openai",
    "model_name": "gpt-6-luna",
    "system_fingerprint": null,
    "id": "chatcmpl-EabPQKUiAU3ZyJM4dSEF9rapcomvJ",
    "service_tier": "default",
    "finish_reason": "stop",
    "logprobs": null
}


### Word Limit
poproś o max N słów, po generacji policz słowa. W razie przekroczenia – skróć odpowiedź.

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-6-luna")

limit = 25
prompt = f"Napisz streszczenie w MAKS {limit} słowach: Czym jest uczenie maszynowe?"

resp = llm.invoke(prompt).content
if len(resp.split()) > limit:
    # szybka korekta – poproś model o skrócenie do limitu
    resp = llm.invoke(f"Skróć to do maks {limit} słów, bez dopisków:\n\n{resp}").content

print(resp)


Uczenie maszynowe to dział sztucznej inteligencji, w którym algorytmy uczą się wzorców z danych i wykorzystują je do przewidywania lub podejmowania decyzji.
